This script fetches Kiel's geographic bounding box from the Nominatim (OpenStreetMap) API and expands it by a given factor to include the surrounding area. The resulting coordinates are saved as a CSV file.

Developed with the assistance of [Claude (Anthropic)](https://claude.ai)

In [1]:
import requests
import polars as pl
import os

BASE_DIR   = "../../data"

def get_kiel_bounding_box(factor=1):
    # Fetches Kiel's bounding box from Nominatim and expands it by the given factor.
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": "Kiel, Germany",
        "format": "json",
        "limit": 1
    }
    headers = {"User-Agent": "Data Science Project CAU WS25/26 Group 16"}

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    bb = data[0]["boundingbox"]
    lat_min = float(bb[0])
    lat_max = float(bb[1])
    lon_min = float(bb[2])
    lon_max = float(bb[3])

    # Increase boundingbox size by "factor"
    lat_center = (lat_min + lat_max) / 2
    lon_center = (lon_min + lon_max) / 2
    lat_span   = (lat_max - lat_min) / 2
    lon_span   = (lon_max - lon_min) / 2

    return {
        "lat_min": lat_center - lat_span * factor,
        "lat_max": lat_center + lat_span * factor,
        "lon_min": lon_center - lon_span * factor,
        "lon_max": lon_center + lon_span * factor,
    }

def save_bounding_box_to_csv(bb):
    # Saves the bounding box dictionary to a CSV file.
    filepath = os.path.join(BASE_DIR, "BoundingBox", "kiel_bb.csv")
    os.makedirs(os.path.join(BASE_DIR, "BoundingBox"), exist_ok=True)
    pl.DataFrame([bb]).write_csv(filepath)
        
bb = get_kiel_bounding_box(factor=3) # Factor 3 to get surrounding of Kiel aswell
save_bounding_box_to_csv(bb)